In [ ]:
# CELL 1

import numpy as np
import polars as pl
from pykalman import KalmanFilter
import importlib
import sys
import os

script_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(script_dir, ".."))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

import config # noqa: E402 # type: ignore

importlib.reload(config)

def calculate_kalman_spread(df_a, df_b, ticker_a, ticker_b):
    valid_start = max(df_a["date"].min(), df_b["date"].min())
    valid_end = min(df_a["date"].max(), df_b["date"].max())

    df_a = df_a.filter((pl.col("date") >= valid_start) & (pl.col("date") <= valid_end))
    df_b = df_b.filter((pl.col("date") >= valid_start) & (pl.col("date") <= valid_end))

    pair_df = df_a.join(df_b, on="date", how="full").sort("date").fill_null(strategy="forward").drop_nulls()
    
    if len(pair_df) < 4000:
        return None, None
    
    log_a = np.log(pair_df[ticker_a].to_numpy())
    log_b = np.log(pair_df[ticker_b].to_numpy())
    
    obs_mat = np.vstack([log_b, np.ones(len(log_b))]).T
    obs_mat = np.expand_dims(obs_mat, axis=1)
    
    trans_cov = config.KALMAN_DELTA / (1 - config.KALMAN_DELTA) * np.eye(2)
    
    kf = KalmanFilter(
        n_dim_obs=1, n_dim_state=2,
        initial_state_mean=np.zeros(2),
        initial_state_covariance=np.ones((2, 2)),
        transition_matrices=np.eye(2),
        transition_covariance=trans_cov,
        observation_matrices=obs_mat,
        observation_covariance=config.KALMAN_OBS_COV
    )
    
    state_means, _ = kf.filter(log_a)
    beta_raw = state_means[:, 0]
    alpha_raw = state_means[:, 1]

    beta = np.concatenate(([beta_raw[0]], beta_raw[:-1]))
    alpha = np.concatenate(([alpha_raw[0]], alpha_raw[:-1]))
    
    spread = log_a - (beta * log_b + alpha)
    
    return spread, pair_df["date"]

# --- MAIN EXECUTION BLOCK FOR CELL 1 ---

print(f"System Mode: {config.SYSTEM_MODE}")

if config.SYSTEM_MODE == "P":
    active_pairs = pl.read_csv("../outputs/active_johansen_pairs.csv")

elif config.SYSTEM_MODE == "R":
    active_pairs = pl.read_csv("../outputs/research_johansen_pairs.csv")
    
print(f"Loaded {len(active_pairs)} active cointegrated pairs from disk.")

pair_spreads = {}

def load_and_clean_data(filepath, ticker_name):

    df = pl.read_csv(filepath, try_parse_dates=True)
    
    df = df.fill_null(strategy="forward")
    
    df = df.with_columns([
        pl.col("close").pct_change().alias("ret_backward")
    ])
    
    df = df.with_columns([
        pl.when(pl.col("ret_backward").abs() > 0.045)
        .then(None)
        .otherwise(pl.col("close"))
        .alias("clean_close")
    ])
    
    df = df.with_columns([
        pl.col("clean_close").forward_fill().alias(ticker_name)
    ])
    
    return df.select(["date", ticker_name])

for row in active_pairs.iter_rows(named=True):
    t_a = row["ticker_a"]
    t_b = row["ticker_b"]
    
    file_a = f"../data/{t_a}_5m.csv"
    file_b = f"../data/{t_b}_5m.csv"
    
    try:
        df_clean_a = load_and_clean_data(file_a, t_a)
        df_clean_b = load_and_clean_data(file_b, t_b)
        
        spread_series, dates = calculate_kalman_spread(df_clean_a, df_clean_b, t_a, t_b)
        
        if spread_series is not None:
            pair_key = f"{t_a}_{t_b}"
            pair_spreads[pair_key] = {
                "spread": spread_series,
                "dates": dates
            }
            print(f"Successfully processed rolling Kalman spread for: {pair_key} ({len(spread_series)} bars)")
    except Exception as e:
        print(f"Failed to process {t_a}/{t_b}: {e}")

System Mode: R
Loaded 21 active cointegrated pairs from disk.
Successfully processed rolling Kalman spread for: RSG_WM (111289 bars)
Successfully processed rolling Kalman spread for: DHR_TMO (111440 bars)
Successfully processed rolling Kalman spread for: EXR_PSA (111462 bars)
Successfully processed rolling Kalman spread for: GS_MS (110676 bars)
Successfully processed rolling Kalman spread for: FCX_NEM (110994 bars)
Successfully processed rolling Kalman spread for: BJ_DLTR (110994 bars)
Successfully processed rolling Kalman spread for: MCO_SPGI (111382 bars)
Successfully processed rolling Kalman spread for: MA_V (109425 bars)
Successfully processed rolling Kalman spread for: CAT_DE (111228 bars)
Successfully processed rolling Kalman spread for: DG_WMT (110994 bars)
Successfully processed rolling Kalman spread for: BAC_PNC (111228 bars)
Successfully processed rolling Kalman spread for: CME_ICE (111462 bars)
Successfully processed rolling Kalman spread for: C_WFC (111618 bars)
Successfull

In [2]:
# CELL 2

import concurrent.futures
import multiprocessing
import polars as pl
from math_engine import process_window 

if __name__ == '__main__':
    jobs = []
    for pair_key, pair_dict in pair_spreads.items():
        ticker_a, ticker_b = pair_key.split("_")
        spread_data = pair_dict["spread"]
        pair_dates = pair_dict["dates"]
        total_length = len(spread_data)
        
        for i in range(config.LOOKBACK_BARS, total_length, config.STEP_BARS):
            jobs.append({
                "pair_key": pair_key,
                "ticker_a": ticker_a,
                "ticker_b": ticker_b,
                "spread_data": spread_data[i - config.LOOKBACK_BARS : i],
                "effective_date": pair_dates[i]
            })

    historical_parameters = []
    cpu_cores = multiprocessing.cpu_count()

    print(f"System Mode: {config.SYSTEM_MODE}")
    print(f"Packaging {len(jobs)} total jobs.")

    with concurrent.futures.ProcessPoolExecutor(max_workers=cpu_cores) as executor:
        results = executor.map(process_window, jobs)
        for result in results: 
            historical_parameters.append(result)

    portfolio_df = pl.DataFrame(historical_parameters).sort(["ticker_a", "ticker_b", "effective_date"])
    portfolio_df.write_csv("../outputs/portfolio_model_parameters.csv")
    print(f"Saved {len(portfolio_df)} rolling parameter regimes to disk.")

System Mode: R
Packaging 14291 total jobs.
Saved 14291 rolling parameter regimes to disk.
